In [3]:
import ROOT as rt
import sys
import collections
from collections import OrderedDict
import uproot
import pandas as pd


import scipy
import awkward
import time

import subprocess
GIT_REPO = subprocess.Popen(['git', 'rev-parse', '--show-toplevel'], stdout=subprocess.PIPE).communicate()[0].rstrip().decode('utf-8')
sys.path.append(GIT_REPO + '/lib/')
from histo_utilities import create_TH1D, create_TH2D, std_color_list, create_TGraph, make_ratio_plot
from helper_functions import *

import numpy as np
from scipy.stats import norm
import math
import CMS_lumi, tdrstyle
style = tdrstyle.setTDRStyle()
CMS_lumi.writeExtraText = 0


# donotdelete = []
print(sys.version)

Welcome to JupyROOT 6.24/06
3.6.8 (default, Nov  2 2021, 13:01:57) 
[GCC 8.4.1 20200928 (Red Hat 8.4.1-1)]


In [4]:
%%time

fpath =OrderedDict()
tree = OrderedDict()
accep_cscdt = OrderedDict()
accep_csccsc = OrderedDict()
start_t = time.time()

# for data_year in ['2022','2023','all']:
version = 'v25'
LUMI_SCALE = 3.2
# for data_year in ['all']:

#     # path = "/storage/af/user/christiw/login-1/christiw/LLP/Run3/CMSSW_14_0_1/src/mds_analysis/data/raw/"
#     # fpath_bkg[data_year] = path + f"data_{data_year}_goodLumi.root"
#     if data_year == 'all':path = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19//Data_all/{version}/normalized/'
#     else: path = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19//Data{data_year}/{version}/normalized/'
#     if data_year == '2022':fpath['data'+data_year] = path + "DisplacedJet-EXOCSCCluster_Run2022-PromptReco_goodLumi.root"
#     elif data_year == '2023':fpath['data'+data_year] = path + "Muon-EXOCSCCluster_Run2023-PromptReco_goodLumi.root"
#     elif data_year == '2024':fpath['data'+data_year] = path + "Muon-Run2024-PromptReco_goodLumi.root"
#     elif data_year == 'all':fpath['data'+data_year] = path + "EXOCSCCluster_Run2022_2024_goodLumi.root"

# for m in [15, 40]:
    
# # fpath['sig'] = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_all/{version}/normalized/ggH_Hto2Sto4B_MH-125-MS-15-ctauS-1000_TuneCP5_13p6TeV_powheg-pythia8_50000pb_weighted.root'
#     fpath[f'{m}'] = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_all/{version}/normalized/ggH_Hto2Sto4D_MH-125-MS-{m}-ctauS-1000_TuneCP5_13p6TeV_powheg-pythia8_50000pb_weighted.root'
# fpath['sig_tau'] = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_all/{version}/normalized/ggH_Hto2Sto4Tau_MH-125-MS-15-ctauS-1000_TuneCP5_13p6TeV_powheg-pythia8_50000pb_weighted.root'
fpath['sig_d'] = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_all/v24/normalized/ggH_Hto2Sto4D_MH-125-MS-15-ctauS-1000_TuneCP5_13p6TeV_powheg-pythia8_50000pb_weighted.root'
fpath['HV_1'] = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_Summer22EE/v25/normalized/HiddenValley_higgs_m_15_ctau_1000_xiO_2p5_xiL_1_23020pb_weighted.root'
fpath['HV_10'] = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_Summer22EE/v25/normalized/HiddenValley_higgs_m_15_ctau_10000_xiO_2p5_xiL_1_23020pb_weighted.root'
# fpath['HV_26'] = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_Summer22EE/v26/normalized/HiddenValley_higgs_m_15_ctau_1000_xiO_2p5_xiL_1_23020pb_weighted.root'
# fpath['HV_10'] = f'/storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_Summer22EE/{version}/normalized/HiddenValley_higgs_m_15_ctau_10000_xiO_2p5_xiL_1_23020pb_weighted.root'

NEvents = {}

for k,v in fpath.items():
    print (k, v)
    root_dir = uproot.open(v) 

    tree[k] = root_dir['MuonSystem']
    NEvents[k] = root_dir['NEvents'].values()[0]
    if not "data" in k:
        accep_csccsc[k] = root_dir['accep_csccsc'].values()[0]
        accep_cscdt[k] = root_dir['accep_cscdt'].values()[0]
    # NEvents[k] = root_dir['NEvents'].values()[0]
    # NEvents[k] = root_dir['NEvents'].counts()
    print("NEvents",NEvents[k])


sig_d /storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_all/v24/normalized/ggH_Hto2Sto4D_MH-125-MS-15-ctauS-1000_TuneCP5_13p6TeV_powheg-pythia8_50000pb_weighted.root
NEvents 57912584.0
HV_1 /storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_Summer22EE/v25/normalized/HiddenValley_higgs_m_15_ctau_1000_xiO_2p5_xiL_1_23020pb_weighted.root
NEvents 46315424.0
HV_10 /storage/af/group/phys_exotica/delayedjets/displacedJetMuonAnalyzer/Run3/V1p19/MC_Summer22EE/v25/normalized/HiddenValley_higgs_m_15_ctau_10000_xiO_2p5_xiL_1_23020pb_weighted.root
NEvents 46301900.0
CPU times: user 713 ms, sys: 10.9 ms, total: 724 ms
Wall time: 722 ms


# CSC-CSC cutflow for AN

In [5]:
%%time
CSC = 'cscRechitCluster'
DT = 'dtRechitCluster'

MET_CUT = 200
MET_THRESHOLD = 50


HIT_THRESHOLD_CSCCSC = { #cscnhit, dtnhit, dphi
    'low MET': (160, 2.8),
    'high MET':(140,1.5),
}

name = [
# "acceptance",
"nClusters == 2",
"HLTDecision",
"Apply L1 efficiency weight",
f"MET < {MET_CUT}",
"noise filter",
"jet veto",
"muon veto",
"ME1 veto",
"time spread",
"time",
"DNN",
"deltaPhi",
"CSC Hit",
    ]
table = {} # one table per sample, one table containing both low MET and high MET
for k,T in tree.items():
    # if  'data' in k:continue
    table[k] = {
    "Selections": [],
    "cut eff (low MET)":[],
    "overall eff (low MET)": [],
    "Nevents (low MET)": [],
    "cut eff (high MET)":[],
    "overall eff (high MET)": [],
    "Nevents (high MET)": [],
    }
    for cat in ['low MET', 'high MET']:
        # print("************************************")
        # print(k, cat)
        # print("************************************")

        if 'data' in k: 
            weight = T['weight'].array()
            total = NEvents[k]
        else: 
            weight = T['pileupWeight'].array()*T['weight'].array()*160./50
            if "HV" in k: weight = 52.2*160000/len(T['weight'].array())+ 0*T['weight'].array()
            total = 52.2*50000*LUMI_SCALE
            temp = 52.2*50000*LUMI_SCALE
            accep = accep_cscdt[k]

        cluster_pass_HLT = HLT_CSC(T['cscRechitClusterEta'].array(),T['cscRechitClusterNStation10'].array(),T['cscRechitClusterSize'].array())
        presel_csccsc = {
           "ncluster": (T["nCscRechitClusters"].array() == 2),
            "HLT": T['HLT_CSCCSC'].array() & (np.sum(cluster_pass_HLT, axis = 1) >= 1) # at least one passes HLT
        }
        # prsel_cscdt = {
        #     "ncluster": (T["nCscRechitClusters"].array() == 1) & (T["nDtRechitClusters"].array() == 1),
        #     "HLT": T['HLT_CSCCSC'].array(),            }

        preselections = presel_csccsc
        for i, sel_k in enumerate(preselections.keys()):
            if i == 0:presel = preselections[sel_k]
            else: presel = presel & preselections[sel_k]
            if "low" in cat: table[k]['Selections'].append(sel_k)
            table[k][f'cut eff ({cat})'].append(np.sum(weight[presel])/temp)
            table[k][f'overall eff ({cat})'].append(np.sum(weight[presel])/total)
            table[k][f'Nevents ({cat})'].append(np.sum(weight[presel]))
            temp = np.sum(weight[presel])
        ### apply L1 efficiency
        if not "data" in k:
            #apply L1 efficiency
            HMTEff = T['cscRechitClusterHMTEfficiency'].array()
            HMTEff = 1-HMTEff[cluster_pass_HLT]
            HMTEff = 1-np.prod(HMTEff,axis=1)
            weight = weight[presel] * HMTEff[presel]
        else: weight = weight[presel]
        if "low" in cat: table[k]['Selections'].append("L1 weight")
        table[k][f'cut eff ({cat})'].append(np.sum(weight)/temp)
        table[k][f'overall eff ({cat})'].append(np.sum(weight)/total)
        table[k][f'Nevents ({cat})'].append(np.sum(weight))

        # dPhi = deltaPhi(np.array(T['cscRechitClusterPhi'].array()[presel][:,0]), np.array(T['dtRechitClusterPhi'].array()[presel][:,0]))

        denominator = np.sum(weight)
        temp = np.sum(weight)
        
        me1 = (T['cscRechitClusterNRechitChamberPlus11'].array()+ T['cscRechitClusterNRechitChamberPlus12'].array()+\
    T['cscRechitClusterNRechitChamberMinus11'].array()+ T['cscRechitClusterNRechitChamberMinus12'].array())
        muonveto = np.logical_not((T['cscRechitClusterMuonVetoPt'].array() > 30) & T['cscRechitClusterMuonVetoGlobal'].array())   
        time = (T['cscRechitClusterTimeWeighted'].array()< 12.5) & (T['cscRechitClusterTimeWeighted'].array()>-5)
        
        cscRechitClusterPhi0 = T['cscRechitClusterPhi'].array()[presel][:,0]
        cscRechitClusterPhi1 = T['cscRechitClusterPhi'].array()[presel][:,1]
        cscRechitClusterEta = T['cscRechitClusterEta'].array()[presel]
        cscRechitClusterNStation = T['cscRechitClusterNStation10'].array()[presel]
        deltaPhi_cluster = deltaPhi(np.array(cscRechitClusterPhi0), np.array(cscRechitClusterPhi1))    
        deltaR_cluster = ((cscRechitClusterEta[:,0]-cscRechitClusterEta[:,1])**2 + deltaPhi_cluster**2)**0.5
        cscRechitClusterSize1 = T['cscRechitClusterSize'].array()[presel][:,1]
        selections = {
             "met $<$ 200 GeV": (T['Puppimet'].array() < 200)[presel],
            f"met $<$ {MET_THRESHOLD} GeV": (T['Puppimet'].array() < MET_THRESHOLD)[presel],
            f"met $>$ {MET_THRESHOLD} GeV": (T['Puppimet'].array() >= MET_THRESHOLD)[presel],
             "noise filter": (T['Flag_all'].array() & T['jetVeto'].array() & T['Flag_ecalBadCalibFilter'].array())[presel],
            "cosmic veto": (T['nCscRings'].array()+T['nDtRings'].array()<10)[presel],
            "jet veto": (np.sum((T[f'{CSC}JetVetoPt'].array()<30)[presel], axis = 1)==2),
            "muon veto": (np.sum(muonveto[presel], axis = 1)==2),
            "ME1/MB1 veto":  (np.sum(me1[presel]==0, axis = 1)==2),
            "CSC time spread": (np.sum((T['cscRechitClusterTimeSpreadWeightedAll'].array()<20)[presel], axis = 1)==2),
            "CSC time": (np.sum(time[presel], axis = 1)==2),
            "CSC DNN": (np.sum(T['cscRechitClusterDNN_bkgMC_plusBeamHalo'].array()[presel]>0.96, axis = 1)==2),
            "Nstation>1": (np.sum(T['cscRechitClusterNStation10'].array()[presel],axis=1) > 2),
            "deltaR": (np.abs(deltaR_cluster)<3.5),
            "deltaPhi": (np.abs(deltaPhi_cluster)>HIT_THRESHOLD_CSCCSC[cat][1]),
            "CSC size": (cscRechitClusterSize1 >= HIT_THRESHOLD_CSCCSC[cat][0]),

        }

        for i, sel_k in enumerate(selections.keys()):
            if i == len(selections.keys())-1 and 'data' in k:continue
            if 'high' in cat and f"met $<$ {MET_THRESHOLD}" in sel_k:continue
            if 'low' in cat and f"met $>$ {MET_THRESHOLD}" in sel_k:continue
            if i == 0:sel = selections[sel_k]
            else: sel = sel & selections[sel_k]

            if "low MET" == cat: 
                if "met" in sel_k and "200" not in sel_k:  table[k]['Selections'].append(f"met $>=$/$<$ {MET_THRESHOLD}")
                else:table[k]['Selections'].append(sel_k)
            table[k][f'cut eff ({cat})'].append(np.sum(weight[sel])/temp)
            table[k][f'overall eff ({cat})'].append(np.sum(weight[sel])/denominator)
            table[k][f'Nevents ({cat})'].append(np.sum(weight[sel]))
            
            temp = np.sum(weight[sel])
    for col in table[k].keys():
        if "eff" in col:
            table[k][col] = np.array(table[k][col])*100
            
    table[k] = pd.DataFrame(table[k])
    latex_table = table[k].to_latex(
    index=False,  # To not include the DataFrame index as a column in the table
    header = ['','cut eff', 'overall eff','Nevents', 'cut eff', 'overall eff','Nevents'],
    column_format="c|c|c|c|c",  # The format of the columns: left-aligned with vertical lines between them
    escape=False,  # Disable escaping LaTeX special characters in the DataFrame
    float_format="{:0.2f}".format,  # Formats floats to two decimal places
    caption = f"Signal Efficiency(\%) of each cut in CSC-CSC category for twin higgs model, LLP mass {k} GeV, lifetime of 1000 mm and decaying to 2 d quarks.\
    Overall Efficiency for all cuts applied after L1 weights are calculated with respect to events after L1 weight and cut efficiency is calculated with respect to the previous cut. ",
    label = f"tab:selections_sig_{k}_1000_dd",
    # position = "!h",
    )

    str_output = latex_table.replace('midrule', 'hline')
    str_output.replace("\toprule","& \multicolumn{2}{c}{low MET} & \multicolumn{2}{c}{high MET} \\")

    print(str_output)


\begin{table}
\centering
\caption{Signal Efficiency(\%) of each cut in CSC-CSC category for twin higgs model, LLP mass sig_d GeV, lifetime of 1000 mm and decaying to 2 d quarks.    Overall Efficiency for all cuts applied after L1 weights are calculated with respect to events after L1 weight and cut efficiency is calculated with respect to the previous cut. }
\label{tab:selections_sig_sig_d_1000_dd}
\begin{tabular}{c|c|c|c|c}
\toprule
                 & cut eff & overall eff &   Nevents & cut eff & overall eff &   Nevents \\
\hline
        ncluster &    1.59 &        1.59 & 132500.70 &    1.59 &        1.59 & 132500.70 \\
             HLT &   39.70 &        0.63 &  52600.49 &   39.70 &        0.63 &  52600.49 \\
       L1 weight &   79.28 &        0.50 &  41703.55 &   79.28 &        0.50 &  41703.55 \\
 met $<$ 200 GeV &   98.48 &       98.48 &  41070.10 &   98.48 &       98.48 &  41070.10 \\
 met $>=$/$<$ 50 &   68.37 &       67.33 &  28080.49 &   31.63 &       31.15 &  12989.61 \\
   

## CSC-DT (combining CSC/DT selections)

In [6]:
%%time
CSC = 'cscRechitCluster'
DT = 'dtRechitCluster'

MET_CUT = 200
MET_THRESHOLD = 50

HIT_THRESHOLD_CSCDT = { #cscnhit, dtnhit, dphi
    'low MET': (160, 130, 2.6),
    'high MET':(120, 100, 2.1),
}
HIT_THRESHOLD_CSCCSC = { #cscnhit, dtnhit, dphi
    'low MET': (160, 2.8),
    'high MET':(140,1.5),
}

name = [
# "acceptance",
"nClusters == 2",
"HLTDecision",
"Apply L1 efficiency weight",
f"MET < {MET_CUT}",
"noise filter",
"jet veto",
"muon veto",
"ME1 veto",
"time spread",
"time",
"DNN",
"deltaPhi",
"CSC Hit",
    ]
table = {} # one table per sample, one table containing both low MET and high MET
for k,T in tree.items():
    if  'data' in k:continue
    
    table[k] = {
    "Selections": [],
    "cut eff (low MET)":[],
    "overall eff (low MET)": [],
    "Nevents (low MET)": [],
    "cut eff (high MET)":[],
    "overall eff (high MET)": [],
    "Nevents (high MET)": [],
    }
   
    for cat in ['low MET', 'high MET']:
        # print("************************************")
        # print(k, cat)
        # print("************************************")

        if 'data' in k: 
            weight = T['weight'].array()
            total = NEvents[k]
        else: 
            weight = T['pileupWeight'].array()*T['weight'].array()*LUMI_SCALE
            if 'HV' in k: weight = T['pileupWeight'].array()*T['weight'].array()*170./23.020
            total = 52.2*50000*LUMI_SCALE
            accep = accep_cscdt[k]
            temp = 52.2*50000*LUMI_SCALE
        cluster_pass_HLT = HLT_CSC(T['cscRechitClusterEta'].array(),T['cscRechitClusterNStation10'].array(),T['cscRechitClusterSize'].array())
        presel_csccsc = {
           "ncluster": (T["nCscRechitClusters"].array() == 2),
            "HLT": T['HLT_CSCCSC'].array() & (np.sum(cluster_pass_HLT, axis = 1) >= 1) # at least one passes HLT
        }
        prsel_cscdt = {
            "ncluster": (T["nCscRechitClusters"].array() == 1) & (T["nDtRechitClusters"].array() == 1),
            "HLT": T['HLT_CSCDT'].array(),            }

        
        preselections = prsel_cscdt
        for i, sel_k in enumerate(preselections.keys()):
            if i == 0:presel = preselections[sel_k]
            else: presel = presel & preselections[sel_k]
            if "low" in cat: table[k]['Selections'].append(sel_k)
            table[k][f'cut eff ({cat})'].append(np.sum(weight[presel])/temp)
            table[k][f'overall eff ({cat})'].append(np.sum(weight[presel])/total)
            table[k][f'Nevents ({cat})'].append(np.sum(weight[presel]))
            temp = np.sum(weight[presel])
        ### apply L1 efficiency
        if not "data" in k:
            if "CSCCSC" in cat:
                #apply L1 efficiency
                HMTEff = T['cscRechitClusterHMTEfficiency'].array()
                HMTEff = 1-HMTEff[cluster_pass_HLT]
                HMTEff = 1-np.prod(HMTEff,axis=1)
                weight = weight[presel] * HMTEff[presel]
            else:
                HMTEff = T['cscRechitClusterHMTEfficiency'].array()[presel][:,0]
                weight = weight[presel] * HMTEff
        else: weight = weight[presel]
        if "low" in cat: table[k]['Selections'].append("L1")
        table[k][f'cut eff ({cat})'].append(np.sum(weight)/temp)
        table[k][f'overall eff ({cat})'].append(np.sum(weight)/total)
        table[k][f'Nevents ({cat})'].append(np.sum(weight))
        print(len(weight))
        # denominator = temp
        temp = np.sum(weight)
        denominator =np.sum(weight)
        dPhi = deltaPhi(np.array(T['cscRechitClusterPhi'].array()[presel][:,0]), np.array(T['dtRechitClusterPhi'].array()[presel][:,0]))
        
        ###cosmic veto for DT clusters###
            
        sel_cosmic = np.logical_and(T['dtRechitClusterNOppositeSegStation1'].array()>0, T['dtRechitClusterNOppositeSegStation2'].array()>0)
        sel_cosmic = np.logical_and(sel_cosmic, T['dtRechitClusterNOppositeSegStation3'].array()>0)
        sel_cosmic = np.logical_and(sel_cosmic, T['dtRechitClusterNOppositeSegStation4'].array()>0)
        sel_cosmic = np.logical_and(sel_cosmic, T['dtRechitClusterNOppositeSegStation1'].array()+T['dtRechitClusterNOppositeSegStation2'].array()+\
                                   T['dtRechitClusterNOppositeSegStation3'].array()+T['dtRechitClusterNOppositeSegStation4'].array()>=6)
        nstation = (T['dtRechitClusterNSegStation1'].array()>1)*1+(T['dtRechitClusterNSegStation2'].array()>1)*1\
        +(T['dtRechitClusterNSegStation3'].array()>1)*1+(T['dtRechitClusterNSegStation4'].array()>1)*1
        sel_cosmic = np.logical_not(np.logical_and(nstation>=3, sel_cosmic))


        selections = {
            "met $<$ 200 GeV": (T['Puppimet'].array() < 200)[presel],
            f"met $<$ {MET_THRESHOLD} GeV": (T['Puppimet'].array() < MET_THRESHOLD)[presel],
            f"met $>$ {MET_THRESHOLD} GeV": (T['Puppimet'].array() >= MET_THRESHOLD)[presel],
            "noise filter": (T['Flag_all'].array() & T['jetVeto'].array() & T['Flag_ecalBadCalibFilter'].array())[presel],
            "cosmic veto": sel_cosmic[presel][:,0] & (T['nCscRings'].array()+T['nDtRings'].array()<10)[presel],
            ### CSC cluster-level selections ###
            "jet veto": ((T[f'{CSC}JetVetoPt'].array()<30)[presel][:,0]) & ((T[f'{DT}JetVetoPt'].array()<30)[presel][:,0]),
            "muon veto": \
            (np.logical_not((T[f'{CSC}MuonVetoPt'].array() > 30) & T[f'{CSC}MuonVetoGlobal'].array())[presel][:,0]) &\
            (np.logical_not((T[f'{DT}MuonVetoPt'].array() > 30) & T[f'{DT}MuonVetoLooseId'].array())[presel][:,0]),
            "ME1/MB1 veto": \
            (((T[f'{CSC}NRechitChamberPlus11'].array()+ T[f'{CSC}NRechitChamberPlus12'].array()+\
            T[f'{CSC}NRechitChamberMinus11'].array()+ T[f'{CSC}NRechitChamberMinus12'].array())==0)[presel][:,0]) & \
            ((T[f'{DT}NHitStation1'].array()==0)[presel][:,0]),
            "CSC time spread": (T[f'{CSC}TimeSpreadWeightedAll'].array()<20)[presel][:,0],
            "CSC time": (T[f'{CSC}TimeWeighted'].array()< 12.5)[presel][:,0] & (T[f'{CSC}TimeWeighted'].array()>-5)[presel][:,0],
            "CSC DNN": (T[f'{CSC}DNN_bkgMC_plusBeamHalo'].array()>0.96)[presel][:,0],
            f"CSC Nhits $>$ {HIT_THRESHOLD_CSCDT[cat][0]}": (T[f'{CSC}Size'].array()>HIT_THRESHOLD_CSCDT[cat][0])[presel][:,0],

#             ### DT cluster-level selections ###
            "DT RPC hit":(T[f'{DT}_match_RPChits_dPhi0p5'].array() >= 1)[presel][:,0],
            "DT BX":(T[f'{DT}_match_RPCBx_dPhi0p5'].array() == 0)[presel][:,0],
            "deltaPhi": np.abs(dPhi) >= HIT_THRESHOLD_CSCDT[cat][2],
            "DT Nhits":(T[f'{DT}Size'].array() >= HIT_THRESHOLD_CSCDT[cat][1])[presel][:,0],
        }
            
        for i, sel_k in enumerate(selections.keys()):
            if i == len(selections.keys())-1 and 'data' in k:continue
            if 'high' in cat and f"met $<$ {MET_THRESHOLD}" in sel_k:continue
            if 'low' in cat and f"met $>$ {MET_THRESHOLD}" in sel_k:continue
            if i == 0:sel = selections[sel_k]
            else: sel = sel & selections[sel_k]
            # print('\t'.join(str(x) for x in [sel_k,  np.sum(weight[sel]), np.sum(weight[sel])/temp]))


            if "low MET" == cat: 
                if "met" in sel_k and "200" not in sel_k:  table[k]['Selections'].append(f"met $>=$/$<$ {MET_THRESHOLD}")
                else:table[k]['Selections'].append(sel_k)
            table[k][f'cut eff ({cat})'].append(np.sum(weight[sel])/temp)
            table[k][f'overall eff ({cat})'].append(np.sum(weight[sel])/denominator)
            table[k][f'Nevents ({cat})'].append(np.sum(weight[sel]))

            temp = np.sum(weight[sel])
            if "cosmic" in sel_k:print(k, np.count_nonzero(sel))
            if i == len(selections.keys())-1:print(cat, k, np.sum(weight[sel]), np.count_nonzero(sel))
    for col in table[k].keys():
        if "eff" in col:
            table[k][col] = np.array(table[k][col])*100
    # print("here")
    table[k] = pd.DataFrame(table[k])
    # print(table[k])
    latex_table = table[k].to_latex(
    index=False,  # To not include the DataFrame index as a column in the table
    header = ['','cut eff', 'overall eff','Nevents','cut eff', 'overall eff','Nevents'],
    column_format="c|c|c|c|c",  # The format of the columns: left-aligned with vertical lines between them
    escape=False,  # Disable escaping LaTeX special characters in the DataFrame
    float_format="{:0.2f}".format,  # Formats floats to two decimal places
    caption = f"Signal Efficiency(\%) of each cut in CSC-DT category for twin higgs model, LLP mass {k} GeV, lifetime of 1000 mm and decaying to 2 d quarks.\
    Overall Efficiency for all cuts applied after L1 weights are calculated with respect to events after L1 weight and cut efficiency is calculated with respect to the previous cut. ",
    label = f"tab:selections_sig_{k}_1000_dd",
    # position = "!h",
    )

    str_output = latex_table.replace('midrule', 'hline')
    str_output.replace("\toprule","& \multicolumn{2}{c}{low MET} & \multicolumn{2}{c}{high MET} \\")

    print(str_output)


6903
sig_d 4565
low MET sig_d 1434.8411 538
6903
sig_d 2000
high MET sig_d 598.1451 222
\begin{table}
\centering
\caption{Signal Efficiency(\%) of each cut in CSC-DT category for twin higgs model, LLP mass sig_d GeV, lifetime of 1000 mm and decaying to 2 d quarks.    Overall Efficiency for all cuts applied after L1 weights are calculated with respect to events after L1 weight and cut efficiency is calculated with respect to the previous cut. }
\label{tab:selections_sig_sig_d_1000_dd}
\begin{tabular}{c|c|c|c|c}
\toprule
                   & cut eff & overall eff &   Nevents & cut eff & overall eff &   Nevents \\
\hline
          ncluster &    2.63 &        2.63 & 219775.42 &    2.63 &        2.63 & 219775.42 \\
               HLT &   10.80 &        0.28 &  23741.84 &   10.80 &        0.28 &  23741.84 \\
                L1 &   73.00 &        0.21 &  17332.27 &   73.00 &        0.21 &  17332.27 \\
   met $<$ 200 GeV &   97.64 &       97.64 &  16923.17 &   97.64 &       97.64 &  16923.17 \

## CSC-DT (splitting CSC/DT selections)

In [7]:
T['weight'].array()

<Array [0.601, 0.601, 0.601, ... 0.601, 0.601] type='1999600 * float32'>

In [5]:
%%time
CSC = 'cscRechitCluster'
DT = 'dtRechitCluster'

MET_CUT = 200
MET_THRESHOLD = 50

HIT_THRESHOLD = { #cscnhit, dtnhit, dphi
    'lowMET': (160, 130, 2.5),
    'highMET':(80, 130, 2.7),
}



name = [
"nCscCluster = 1, nDtCluster = 1",
"HLTDecision",
"Offline HLT cut",
"Apply L1 efficiency weight",
f"MET < {MET_CUT}",
"noise filter",
"jet veto",
"muon veto",
"ME1 veto",
"time spread",
"time",
"DNN",
"deltaPhi",
"CSC Hit",
    ]
for k,T in tree.items():
    if  'data' in k:continue
    for cat in ['lowMET', 'highMET']:
        print("************************************")
        print(k, cat)
        print("************************************")

        if 'data' in k: 
            weight = T['weight'].array()
            total = NEvents[k]
        else: 
            weight = T['pileupWeight'].array()*T['weight'].array()*LUMI_SCALE
            total = 52.2*50000*LUMI_SCALE

        presel =  (T["nCscRechitClusters"].array() == 1) & (T["nDtRechitClusters"].array() == 1)
        print('\t'.join(str(x) for x in [name[0],  np.sum(weight[presel]), np.sum(weight[presel])/total, np.sum(weight[presel]/total)]))
        temp = np.sum(weight[presel])

        presel =  presel & T['HLT_CSCDT'].array()
        print('\t'.join(str(x) for x in [name[1],  np.sum(weight[presel]), np.sum(weight[presel])/temp, np.sum(weight[presel]/total)]))
        temp = np.sum(weight[presel])


        #apply L1 efficiency
        if 'sig' in k:
            HMTEff = T['cscRechitClusterHMTEfficiency'].array()[presel][:,0]
            weight = weight[presel] * HMTEff
        else: weight = weight[presel]
        print('\t'.join(str(x) for x in [name[3],  np.sum(weight), np.sum(weight)/temp, np.sum(weight/total)]))
        temp = np.sum(weight)

        dPhi = deltaPhi(np.array(T['cscRechitClusterPhi'].array()[presel][:,0]), np.array(T['dtRechitClusterPhi'].array()[presel][:,0]))

        selections = {
            "met < 200": (T['met'].array() < 200)[presel],
            f"met < {MET_THRESHOLD}": (T['met'].array() < MET_THRESHOLD)[presel],
            f"met > {MET_THRESHOLD}": (T['met'].array() > MET_THRESHOLD)[presel],
            "noise filter": (T['Flag_all'].array())[presel],
            ### CSC cluster-level selections ###

            "CSC jet veto": (T[f'{CSC}JetVetoPt'].array()<30)[presel][:,0],
            "CSC muon veto": np.logical_not((T[f'{CSC}MuonVetoPt'].array() > 30) & T[f'{CSC}MuonVetoGlobal'].array())[presel][:,0],
            "CSC ME1 veto": ((T[f'{CSC}NRechitChamberPlus11'].array()+ T[f'{CSC}NRechitChamberPlus12'].array()+\
            T[f'{CSC}NRechitChamberMinus11'].array()+ T[f'{CSC}NRechitChamberMinus12'].array())==0)[presel][:,0],
            "CSC time spread": (T[f'{CSC}TimeSpreadWeightedAll'].array()<20)[presel][:,0],
            "CSC time cut": (T[f'{CSC}TimeWeighted'].array()< 12.5)[presel][:,0] & (T[f'{CSC}TimeWeighted'].array()>-5)[presel][:,0],
            "CSC DNN": (T[f'{CSC}DNN_bkgMC_plusBeamHalo'].array()>0.96)[presel][:,0],
            f"CSC Nhits > {HIT_THRESHOLD[cat][0]}": (T[f'{CSC}Size'].array()>HIT_THRESHOLD[cat][0])[presel][:,0],

            ### DT cluster-level selections ###
            "DT jet veto":(T[f'{DT}JetVetoPt'].array()<30)[presel][:,0],
            "DT muon veto":np.logical_not((T[f'{DT}MuonVetoPt'].array() > 30) & T[f'{DT}MuonVetoLooseId'].array())[presel][:,0],
            "DT MB1 veto":(T[f'{DT}NHitStation1'].array()==0)[presel][:,0],
            "DT RPC hit":(T[f'{DT}_match_RPChits_dPhi0p5'].array() >= 1)[presel][:,0],
            "DT BX":(T[f'{DT}_match_RPCBx_dPhi0p5'].array() == 0)[presel][:,0],
            f"deltaPhi >= {HIT_THRESHOLD[cat][2]}": np.abs(dPhi) >= HIT_THRESHOLD[cat][2],
            f"DT Nhits >= {HIT_THRESHOLD[cat][1]}":(T[f'{DT}Size'].array() >= HIT_THRESHOLD[cat][1])[presel][:,0],
        }

        for i, sel_k in enumerate(selections.keys()):
            if i == len(selections.keys())-1 and 'data' in k:continue
            if 'high' in cat and f"met < {MET_THRESHOLD}" in sel_k:continue
            if 'low' in cat and f"met > {MET_THRESHOLD}" in sel_k:continue
            if i == 0:sel = selections[sel_k]
            else: sel = sel & selections[sel_k]
            print('\t'.join(str(x) for x in [sel_k,  np.sum(weight[sel]), np.sum(weight[sel])/temp, np.sum(weight[sel]/total)]))
            temp = np.sum(weight[sel])



************************************
sig_b lowMET
************************************
nCscCluster = 1, nDtCluster = 1	210574.0	0.028408342776968323	0.028408343
HLTDecision	21212.11	0.1007347	0.002861706
Apply L1 efficiency weight	14901.752	0.70251155	0.0020103815
met < 200	14589.773	0.9790643	0.001968293
met < 50	8281.689	0.5676366	0.0011172751
noise filter	8281.689	1.0	0.0011172751
CSC jet veto	8147.3613	0.9837801	0.001099153
CSC muon veto	8145.26	0.99974203	0.0010988694
CSC ME1 veto	6814.369	0.8366055	0.0009193202
CSC time spread	6730.3135	0.98766494	0.00090798025
CSC time cut	6730.3135	1.0	0.00090798025
CSC DNN	5913.905	0.8786968	0.00079783937
CSC Nhits > 160	4553.0083	0.7698819	0.00061424216
DT jet veto	4477.9033	0.9835043	0.0006041097
DT muon veto	4477.9033	1.0	0.0006041097
DT MB1 veto	3666.8728	0.81888163	0.0004946944
DT RPC hit	3552.0261	0.96867996	0.00047920056
DT BX	3501.1167	0.9856675	0.00047233238
deltaPhi >= 2.5	2979.5479	0.85102785	0.00040196802
DT Nhits >= 130	1242.2438	

In [3]:
for k, T in tree.items():
    print(NEvents[k], len(T['evtNum'].array()))

57055616.0 169932
58031464.0 93514
57931310.0 159603


In [ ]:
print("test")